In [ ]:
from openai import OpenAI # Importez les bibliothèques nécessaires
import pandas as pd
import json
from tqdm import tqdm

client = OpenAI(
    api_key="" # Remplacez par votre clé API
)

df = pd.read_csv("comments_tiktok.csv") # Chargez les données à partir du fichier CSV contenant les commentaires TikTok 

annotations=[] # Créez une liste pour stocker les annotations

for i, comment in enumerate( # Utilisez tqdm pour afficher une barre de progression pendant l'annotation
    tqdm(
        df["comment_text"],
        total=len(df),
        desc="Annotation linguistique",
        unit="commentaire"
    )
):

    prompt=f"""
Tu es un annotateur linguistique.

Commentaire :
{comment}

Retourne uniquement un JSON :

{{
"abbreviation_words": true/false,
"syntax_suppression": true/false,
"phonetic_writing": true/false,
"emoji_replaces_text": true/false,
"letter_repetition_emotion": true/false
}}

Définitions IMPORTANTES :

1. abbreviation_words

Définition :
Un mot est considéré comme une abréviation lorsqu'un ou plusieurs caractères sont supprimés afin de raccourcir une expression connue.

TRUE :
- mdr
- ptdr
- cmon
- idk
- omg
- tqt
- jsp
- pq

FALSE :
- maison
- incroyable
- bonjour
- slay
- queen


2. syntax_suppression

Définition :
La structure grammaticale normale est simplifiée ou incomplète.

TRUE :
- moi manger pizza
- toi trop drôle
- lui pas comprendre
- need this
- trop beau ce film
- moi quand je vois ça

FALSE :
- Je mange une pizza
- Cette vidéo est drôle
- I need this now


3. phonetic_writing

Définition :
Orthographe modifiée pour reproduire l'oral.

TRUE :
- sava
- keske
- chui
- ta vu
- r u
- u
- ur
- wanna
- gonna

FALSE :
- ça va
- je suis
- you


4. emoji_replaces_text

Définition :
Emoji remplace directement un mot ou une expression.

TRUE :
- je suis 😭
- moi après examen 💀
- ❤️
- 😂😂😂
- il est 🥵

FALSE :
- trop drôle 😂
- j'adore 😭


5. letter_repetition_emotion

Définition :
Répétition de lettres pour augmenter l'intensité émotionnelle.

TRUE :
- mdrrrrrrrr
- nooooo
- slayyyyy
- omgggg
- stoooop

FALSE :
- mdr
- omg
- stop 


Règles générales :

- Plusieurs catégories peuvent être TRUE
- Les catégories sont indépendantes
- En cas de doute → FALSE
- Ne rien inventer
- Utiliser uniquement ce qui est écrit
- Retourner uniquement le JSON
"""

    try: # Envoyez le prompt à l'API et récupérez la réponse

        response = client.responses.create(
            model="gpt-5-mini",
            input=prompt
        )

        result = json.loads(
            response.output_text
        )

    except Exception as e: # En cas d'erreur, affichez un message et utilisez des valeurs par défaut

        print("\nErreur :", e)

        result = {
            "abbreviation_words":False,
            "syntax_suppression":False,
            "phonetic_writing":False,
            "emoji_replaces_text":False,
            "letter_repetition_emotion":False
        }

    annotations.append(result) # Ajoutez les résultats à la liste des annotations


    if (i+1) % 50 == 0: # Toutes les 50 annotations, créez une sauvegarde partielle

        temp_annotations = pd.DataFrame(
            annotations
        )

        temp_final = pd.concat(
            [
                df.iloc[:len(annotations)],
                temp_annotations
            ],
            axis=1
        )

        temp_final.to_csv(
            "backup.csv",
            index=False
        )

        print(
            f"\nBackup créé : {i+1}/{len(df)}"
        )


annotation_df = pd.DataFrame( # Convertissez les annotations en DataFrame
    annotations
)

final = pd.concat( # Combinez les données originales avec les annotations
    [df, annotation_df],
    axis=1
)

final.to_csv( # Sauvegardez le résultat final dans un nouveau fichier CSV
    "comments_annotated.csv",
    index=False
)

print("\nTerminé") # Affichez un message de fin avec le nombre de commentaires annotés et le nom du fichier créé
print(f"{len(final)} commentaires sauvegardés")
print("Fichier créé : comments_annotated.csv")

Annotation linguistique:   2%|▏         | 50/2500 [06:03<4:25:06,  6.49s/commentaire]


Backup créé : 50/2500


Annotation linguistique:   4%|▍         | 100/2500 [13:46<4:11:52,  6.30s/commentaire]


Backup créé : 100/2500


Annotation linguistique:   6%|▌         | 150/2500 [21:06<4:41:38,  7.19s/commentaire] 

...


Backup créé : 2400/2500


Annotation linguistique:  98%|█████████▊| 2450/2500 [4:44:42<07:35,  9.11s/commentaire]   


Backup créé : 2450/2500


Annotation linguistique: 100%|██████████| 2500/2500 [4:49:46<00:00,  6.95s/commentaire]


Backup créé : 2500/2500

Terminé
2500 commentaires sauvegardés
Fichier créé : comments_annotated.csv
